# 05 · 结果收集与表格导出

本 Notebook 只调用 `collect_results.py` 与 `export_tables.py`，不在 UI 层重新统计、选择 best 或计算 mean/std。随后用 pandas 展示正式 CSV，并显式显示 successful/failed seeds。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"
RESULTS_ROOT = ARTIFACT_ROOT / "results"
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)

In [ ]:
collect_command = [
    sys.executable, str(PROJECT_ROOT / "src/experiments/collect_results.py"),
    "--artifact-root", str(ARTIFACT_ROOT),
]
export_command = [
    sys.executable, str(PROJECT_ROOT / "src/experiments/export_tables.py"),
    "--artifact-root", str(ARTIFACT_ROOT),
]
subprocess.run(collect_command, cwd=PROJECT_ROOT, check=True)
subprocess.run(export_command, cwd=PROJECT_ROOT, check=True)

In [ ]:
table_names = [
    "all_runs.csv", "main_results.csv", "ablation_results.csv",
    "calibration_results.csv", "efficiency_results.csv",
]
tables = {}
for name in table_names:
    path = RESULTS_ROOT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    tables[name] = pd.read_csv(path)
    print(f"{name}: {len(tables[name])} rows")
    display(tables[name])

In [ ]:
# 直接展示 export_tables.py 写出的 seed 状态列，不做二次聚合。
for name in ("main_results.csv", "ablation_results.csv", "calibration_results.csv", "efficiency_results.csv"):
    frame = tables[name]
    seed_columns = [
        column for column in (
            "dataset", "experiment", "successful_seeds", "failed_seeds",
            "successful_seed_count", "failed_seed_count",
        ) if column in frame.columns
    ]
    print(name)
    display(frame[seed_columns])

print("all_runs 原始状态（失败不会作为 0 参与统计）：")
display(tables["all_runs.csv"][["dataset", "config", "seed", "status", "failure_type", "failure_message"]])